In [ ]:
# ==========================================
# Download Waterbirds to Google Drive
# ==========================================
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set paths
DRIVE_ROOT = "/content/drive/MyDrive/datasets"
TGZ_PATH = f"{DRIVE_ROOT}/waterbird.tar.gz"
LOCAL_ROOT = "/content/waterbirds_local"

# Final DATA_ROOT will be set after extraction
DATA_ROOT = f"{DRIVE_ROOT}/waterbirds"  # Default - will be updated

# Create directories
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Download using wget (more reliable than urllib for problematic URLs)
if not os.path.exists(TGZ_PATH) and not os.path.exists(DATA_ROOT):
    print("Attempting to download Waterbirds dataset to Google Drive...")

    # Try multiple sources
    sources = [
        "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz",
        "https://worksheets.codalab.org/rest/bundles/0xb922b6149f9043789ae932c02fe39043/contents/blob/",
    ]

    success = False
    for i, url in enumerate(sources):
        print(f"\nTrying source {i+1}...")
        result = os.system(f'wget -O "{TGZ_PATH}" "{url}" 2>&1')
        if result == 0 and os.path.exists(TGZ_PATH) and os.path.getsize(TGZ_PATH) > 1000:
            print("Download successful!")
            success = True
            break
        else:
            print(f"Source {i+1} failed")
            if os.path.exists(TGZ_PATH):
                os.remove(TGZ_PATH)

    if not success:
        print("\n❌ Automatic download failed.")
        print("Please download manually from: https://github.com/kohpangwei/group_DRO")

# Extract if we have the tar.gz and haven't extracted yet
if os.path.exists(TGZ_PATH) and not os.path.exists(DATA_ROOT):
    print("\nExtracting dataset to Google Drive...")
    import tarfile
    with tarfile.open(TGZ_PATH, "r:gz") as tf:
        tf.extractall(DRIVE_ROOT)

    # The tar extracts to waterbird_complete95_forest2water2, rename to waterbirds
    extracted_path = f"{DRIVE_ROOT}/waterbird_complete95_forest2water2"
    if os.path.exists(extracted_path):
        os.rename(extracted_path, DATA_ROOT)
        print(f"✓ Renamed to {DATA_ROOT}")

# Copy to local disk for faster I/O
if os.path.exists(DATA_ROOT) and not os.path.exists(LOCAL_ROOT):
    print(f"\n🚀 Copying dataset to local disk ({LOCAL_ROOT}) for faster I/O...")
    os.makedirs(LOCAL_ROOT, exist_ok=True)

    if os.path.exists(TGZ_PATH):
        # Extract from tar directly to local (faster)
        os.system(f"cp '{TGZ_PATH}' /content/temp_waterbirds.tar.gz")
        os.system(f"tar -xf /content/temp_waterbirds.tar.gz -C '{LOCAL_ROOT}'")
        if os.path.exists("/content/temp_waterbirds.tar.gz"):
            os.remove("/content/temp_waterbirds.tar.gz")
    else:
        # Copy from Drive
        import distutils.dir_util
        distutils.dir_util.copy_tree(DATA_ROOT, LOCAL_ROOT)
    print("✅ Data ready on local disk!")

# Set final DATA_ROOT - prefer local for speed
if os.path.exists(LOCAL_ROOT):
    if "waterbird_complete95_forest2water2" in os.listdir(LOCAL_ROOT):
        DATA_ROOT = os.path.join(LOCAL_ROOT, "waterbird_complete95_forest2water2")
    elif os.path.exists(os.path.join(LOCAL_ROOT, "metadata.csv")):
        DATA_ROOT = LOCAL_ROOT
    else:
        # Check subdirectories
        for item in os.listdir(LOCAL_ROOT):
            subpath = os.path.join(LOCAL_ROOT, item)
            if os.path.isdir(subpath) and os.path.exists(os.path.join(subpath, "metadata.csv")):
                DATA_ROOT = subpath
                break

# Check final status
if os.path.exists(DATA_ROOT) and os.path.exists(os.path.join(DATA_ROOT, "metadata.csv")):
    print(f"\n✅ Waterbirds dataset ready at: {DATA_ROOT}")
    print(f"Contents: {os.listdir(DATA_ROOT)[:5]}...")
else:
    print(f"\n❌ Dataset not properly set up at {DATA_ROOT}")
    print("Please check paths and try again.")

In [ ]:
# Diagnostic: Check what's actually in your dataset
import os
import pandas as pd

# DATA_ROOT should already be set from previous cell
print(f"DATA_ROOT = {DATA_ROOT}")

# Check if basic structure exists
print("\nChecking dataset structure...")
print(f"DATA_ROOT exists: {os.path.exists(DATA_ROOT)}")
print(f"metadata.csv exists: {os.path.exists(os.path.join(DATA_ROOT, 'metadata.csv'))}")

# List contents
if os.path.exists(DATA_ROOT):
    print(f"\nContents of {DATA_ROOT}:")
    contents = os.listdir(DATA_ROOT)
    for item in contents[:10]:
        print(f"  - {item}")

# Check metadata structure
meta_path = os.path.join(DATA_ROOT, "metadata.csv")
if os.path.exists(meta_path):
    meta = pd.read_csv(meta_path)
    print(f"\nMetadata columns: {list(meta.columns)}")
    print(f"Number of rows: {len(meta)}")
    print("\nFirst few img_filename entries:")
    print(meta['img_filename'].head(5))

In [ ]:
# ==========================================
# Verify DATA_ROOT is set correctly
# ==========================================
# DATA_ROOT should already be set from the first cell
# This cell just verifies everything is ready

print(f"Final DATA_ROOT: {DATA_ROOT}")
print(f"Exists: {os.path.exists(DATA_ROOT)}")

if os.path.exists(DATA_ROOT):
    contents = os.listdir(DATA_ROOT)
    print(f"Contents ({len(contents)} items): {contents[:5]}...")

    # Verify key files exist
    meta_exists = os.path.exists(os.path.join(DATA_ROOT, "metadata.csv"))
    print(f"metadata.csv exists: {meta_exists}")

    if meta_exists:
        print("✅ Dataset is ready!")
    else:
        print("❌ metadata.csv not found - check DATA_ROOT path")
else:
    print("❌ DATA_ROOT does not exist")

In [ ]:
# -*- coding: utf-8 -*-
"""
IRS(KL)-Group vs CVaR DRO on Waterbirds (ResNet50)
- Protocol: Strict Train/Val split. Test set only evaluated at the END.
- Monitoring: Prints Train Loss, Validation Worst-Group Acc, and h_bar.
- Optimization: Fast Vectorized Search, Fixed Seeds.
- FIXED: Added missing PIL Image import.
"""

import os
import time
import math
import warnings
import random
import copy
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image  # <--- FIXED: ADDED THIS IMPORT

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

# ==========================================
# 1. CONFIGURATION
# ==========================================
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# >>> UPDATE PATHS <<<
DATA_ROOT = "/content/drive/MyDrive/datasets/waterbirds"

# Hyperparameters
SEED = 42
NUM_CLASSES = 2
NUM_GROUPS = 4
BATCH_SIZE = 256
IMG_SIZE = 224

# Training Settings
EPOCHS = 50
WARMUP_EPOCHS = 1
LR_IRS = 1e-5
LR_DRO = 1e-3
WEIGHT_DECAY = 1e-4

# IRS Parameters
IRS_TAU = 0.1
IRS_DISTANCE_SCALE = 1.0
IRS_MIN_DIV = 1e-2
IRS_H_MAX = 100.0       # prevents insane h
IRS_TAU_EPS = 1e-4      # feasibility margin
IRS_TAU_MULT = 1.01   # your “loss * 1.01 until final tau” rule



# ==========================================
# 2. UTILS
# ==========================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# ==========================================
# 3. DATA & MODEL
# ==========================================
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class WaterbirdsDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.root = root
        self.transform = transform
        meta_path = os.path.join(root, "metadata.csv")
        if not os.path.exists(meta_path): raise FileNotFoundError(f"Metadata not found at {meta_path}")
        meta = pd.read_csv(meta_path)
        split_map = {"train": 0, "val": 1, "test": 2}
        self.meta = meta[meta["split"] == split_map[split]].reset_index(drop=True)
        self.meta["group"] = self.meta["y"] * 2 + self.meta["place"]

    def __len__(self): return len(self.meta)
    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        img_path = os.path.join(self.root, "images", row["img_filename"])
        if not os.path.exists(img_path): img_path = os.path.join(self.root, row["img_filename"])
        img = Image.open(img_path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, int(row["y"]), int(idx), int(row["group"])

def get_loaders_and_priors():
    train_ds = WaterbirdsDataset(DATA_ROOT, "train", train_tf)
    val_ds   = WaterbirdsDataset(DATA_ROOT, "val", eval_tf)
    test_ds  = WaterbirdsDataset(DATA_ROOT, "test", eval_tf)

    group_counts = train_ds.meta["group"].value_counts().sort_index().values
    P_group = group_counts / group_counts.sum()
    P_group_tensor = torch.tensor(P_group, dtype=torch.float32, device=device)

    loaders = {
        "train": DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True),
        "val":   DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True),
        "test":  DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    }
    return loaders, P_group_tensor

def fresh_model():
    weights = ResNet50_Weights.IMAGENET1K_V1
    model = resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(device)

# ==========================================
# 4. IRS MATH
# ==========================================
@torch.no_grad()
def _kappa_for_h(F_vals, P_hat, tau_t, h, distance_scale=1.0, min_div=1e-2):
    log_base = torch.log(P_hat.clamp_min(1e-30))
    logits = log_base + (F_vals * h)
    logits = torch.clamp(logits, min=logits.max()-40.0, max=logits.max()+1e-6)
    P = torch.softmax(logits, dim=0)
    E_F = torch.dot(P, F_vals)
    num = E_F - tau_t
    Dkl = (P * (torch.log(P.clamp_min(1e-30)) - torch.log(P_hat.clamp_min(1e-30)))).sum()
    den = distance_scale * torch.max(Dkl, torch.tensor(0.0, device=device)) + min_div
    kappa = num / torch.max(den, torch.tensor(1e-20, device=device))
    return kappa, num, Dkl, P

@torch.no_grad()
def _fast_secant_search(F_vals, P_hat, tau_t, h_prev,
                        dist_scale=1.0, min_div=1e-2, max_iter=2):
    h0 = torch.tensor(max(0.0, h_prev - 2.0), device=device)
    h1 = torch.tensor(max(0.1, h_prev + 2.0), device=device)
    h0 = torch.clamp(h0, 0.0, IRS_H_MAX)
    h1 = torch.clamp(h1, 0.0, IRS_H_MAX)

    k0, _, _, _ = _kappa_for_h(F_vals, P_hat, tau_t, h0, dist_scale, min_div)
    k1, _, _, _ = _kappa_for_h(F_vals, P_hat, tau_t, h1, dist_scale, min_div)

    for _ in range(max_iter):
        diff = h1 - h0
        slope = (k1 - k0) / (diff + 1e-8)
        h_new = torch.where(torch.abs(slope) > 1e-12, h1 - k1 / slope, h1 + 1.0)
        h_new = torch.max(h_new, torch.tensor(0.0, device=device))
        h_new = torch.clamp(h_new, 0.0, IRS_H_MAX)
        k_new, _, _, _ = _kappa_for_h(F_vals, P_hat, tau_t, h_new, dist_scale, min_div)
        h0 = h1; k0 = k1
        h1 = h_new; k1 = k_new

    _, num, Dkl, P = _kappa_for_h(F_vals, P_hat, tau_t, h1, dist_scale, min_div)
    return h1.item(), num, Dkl, P

def compute_group_stats(ce_vec, groups, num_groups, P_global):
    counts = torch.bincount(groups, minlength=num_groups).float()
    mask = counts > 0

    ce_sum = torch.zeros(num_groups, device=device).scatter_add_(0, groups, ce_vec)
    F = torch.zeros(num_groups, device=device)
    F[mask] = ce_sum[mask] / counts[mask]

    # reference distribution restricted to present groups, renormalized
    P_hatS = torch.zeros_like(F)
    mass_S = P_global[mask].sum().clamp_min(1e-12)
    P_hatS[mask] = P_global[mask] / mass_S

    return F, P_hatS, mask


# ==========================================
# 5. TRAINING
# ==========================================
@torch.no_grad()
def eval_metrics(model, loader):
    model.eval()
    total_loss, total_n = 0., 0
    group_correct = torch.zeros(NUM_GROUPS)
    group_total = torch.zeros(NUM_GROUPS)
    ce_loss = nn.CrossEntropyLoss(reduction='sum')

    for x, y, _, g in loader:
        x, y, g = x.to(device), y.to(device), g.to(device)
        logits = model(x)
        total_loss += ce_loss(logits, y).item()
        preds = logits.argmax(1)
        correct = (preds == y)
        total_n += y.size(0)
        for k in range(NUM_GROUPS):
            mask = (g == k)
            if mask.any():
                group_correct[k] += correct[mask].sum().item()
                group_total[k] += mask.sum().item()
    group_accs = group_correct / group_total.clamp(min=1)
    return total_loss/total_n, group_accs.min().item()

@torch.no_grad()
def estimate_reference_risk_ce(model, loader, P_global_group):
    """
    Estimates R(q_ref) = sum_g P_global[g] * r_g
    where r_g is the mean CE loss for group g over the given loader.
    """
    model.eval()
    ce_none = nn.CrossEntropyLoss(reduction="none")

    sum_loss = torch.zeros(NUM_GROUPS, device=device)
    count = torch.zeros(NUM_GROUPS, device=device)

    for x, y, _, g in loader:
        x, y, g = x.to(device), y.to(device), g.to(device)
        logits = model(x)
        ce = ce_none(logits, y)

        sum_loss.scatter_add_(0, g, ce)
        count.scatter_add_(0, g, torch.ones_like(ce))

    F = sum_loss / count.clamp_min(1.0)  # group mean CE
    return torch.dot(F, P_global_group).item()


def train_algorithm(algo_name, model, loaders, P_global_group, epochs, lr, warmup_epochs=0, alpha=0.2):
    print(f"\n[{algo_name}] Training {epochs} eps...")

    if algo_name == "IRS-Group":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY, momentum=0.9)

    ce_none = nn.CrossEntropyLoss(reduction='none')

    # Track Best Model
    best_val_worst = -1.0
    best_model_state = None

    # 1. Warmup (IRS Only)
    if algo_name == "IRS-Group":
        for ep in range(1, warmup_epochs + 1):
            model.train()
            running_loss = 0.0
            n_batches = 0
            for x, y, _, _ in loaders['train']:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                loss = nn.CrossEntropyLoss()(model(x), y)
                loss.backward(); opt.step()
                running_loss += loss.item()
                n_batches += 1

            # Eval
            v_loss, v_worst = eval_metrics(model, loaders['val'])
            print(f"[Warmup] Ep {ep}: Tr Loss {running_loss/n_batches:.4f} | Val Worst {v_worst:.4f}")


    # 2. Main Loop
    prev_h = 0.0
    for ep in range(warmup_epochs + 1, epochs + 1):
        model.train()
        sum_h, cnt = 0.0, 0
        running_loss = 0.0
        n_batches = 0

        for x, y, idx, g in tqdm(loaders['train'], desc=f"{algo_name} Ep {ep}", leave=False):
            x, y, g = x.to(device), y.to(device), g.to(device)
            opt.zero_grad()
            logits = model(x)
            ce_vec = ce_none(logits, y)

            loss = None
            if algo_name == "IRS-Group":
                # IRS Logic
                F_b, P_b, mask = compute_group_stats(ce_vec, g, NUM_GROUPS, P_global_group)

                m_ref_b = torch.dot(F_b, P_b)              # reference risk on this batch support
                maxF_b  = F_b[mask].max()

                # final target tau (what you ultimately want to satisfice)
                tau_final_t = F_b.new_tensor(float(IRS_TAU))

                # tau used ONLY for root existence / worst-case construction until we reach final tau
                tau_root = max(float(IRS_TAU), float(m_ref_b.item()) * IRS_TAU_MULT)

                # keep tau_root strictly below maxF to avoid degenerate root issues
                tau_root = min(tau_root, float(maxF_b.item()) - IRS_TAU_EPS)

                # if batch is degenerate (all present groups identical so maxF≈m_ref), skip
                if tau_root <= float(m_ref_b.item()) + IRS_TAU_EPS:
                    continue

                tau_root_t = F_b.new_tensor(tau_root)

                try:
                    h_star, num, D_kl, P_star_b = _fast_secant_search(
                        F_b, P_b, tau_root_t, prev_h, IRS_DISTANCE_SCALE, IRS_MIN_DIV, max_iter=2
                    )

                except:
                    h_star = prev_h
                    _, _, D_kl, P_star_b = _kappa_for_h(F_b, P_b, tau_root_t, h_star, IRS_DISTANCE_SCALE, IRS_MIN_DIV)

                r_wc = torch.dot(P_star_b, F_b)  # worst-case (under the constructed distribution)

                # ---- YOUR gating: stop updating once we beat final tau on worst-case ----
                if r_wc <= tau_final_t + IRS_TAU_EPS:
                    continue  # objective is 0, no update

                # otherwise do the robust-weighted CE update
                P_const = P_star_b.detach()
                loss = torch.dot(P_const, F_b)

                sum_h += min(h_star, IRS_H_MAX)
                prev_h = min(h_star, IRS_H_MAX)
                cnt += 1


            else:
                # DRO Logic
                k = max(1, int(alpha * ce_vec.size(0)))
                top_losses, _ = torch.topk(ce_vec, k)
                loss = top_losses.mean()

            loss.backward()
            opt.step()

            running_loss += loss.item()
            n_batches += 1

        # Eval
        val_loss, val_worst = eval_metrics(model, loaders['val'])

        # Checkpoint Best Model based on VALIDATION only
        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())
            is_best = "*NEW BEST*"
        else:
            is_best = ""

        h_info = f"| h_bar: {sum_h/max(1, cnt):.1f}" if algo_name == "IRS-Group" else ""
        avg_tr = running_loss / max(1, n_batches)
        print(f"[{algo_name} Ep {ep:02d}] Tr Loss: {avg_tr:.4f} | Val Worst: {val_worst:.4f} {h_info} {is_best}")


    # Load Best Model for Final Test Evaluation
    model.load_state_dict(best_model_state)
    return model

# ==========================================
# 6. EXECUTION
# ==========================================
'''
if __name__ == "__main__":
    set_seed(SEED)
    print(f"Seed set to {SEED}")
    print("Preparing Data...")
    loaders, P_global_group = get_loaders_and_priors()

    # --------------------------------------
    # RUN 1: IRS-Group
    # --------------------------------------
    print("\n" + "="*40)
    print(" STARTING: IRS-Group")
    print("="*40)
    t0 = time.time()
    set_seed(SEED)
    model_irs = fresh_model()
    # Train and get best model
    best_model_irs = train_algorithm("IRS-Group", model_irs, loaders, P_global_group,
                                     epochs=EPOCHS, lr=LR_IRS, warmup_epochs=WARMUP_EPOCHS)
    time_irs = (time.time() - t0) / 60

    # --------------------------------------
    # FINAL EVALUATION (Rigorous)
    # --------------------------------------
    print("\n" + "="*40)
    print(" FINAL EVALUATION ON TEST SET")
    print(" (Evaluating Best Validation Models)")
    print("="*40)

    _, test_worst_irs = eval_metrics(best_model_irs, loaders['test'])

    print(f"{'Method':<15} | {'Worst-Group Acc':<15} | {'Time (min)':<10}")
    print("-" * 45)
    print(f"{'IRS-Group':<15} | {test_worst_irs:.4f}          | {time_irs:.2f}")

'''

In [ ]:
# ==========================================
# DETAILED METRICS & CONFUSION MATRIX
# ==========================================
@torch.no_grad()
def get_detailed_metrics(model, loader, model_name):
    model.eval()
    group_correct = torch.zeros(NUM_GROUPS)
    group_total = torch.zeros(NUM_GROUPS)
    total_correct = 0
    total_samples = 0

    # Confusion Matrix: Rows = True Label, Cols = Pred Label
    # Class 0: Landbird, Class 1: Waterbird
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)

    for x, y, _, g in loader:
        x, y, g = x.to(device), y.to(device), g.to(device)
        logits = model(x)
        preds = logits.argmax(1)

        correct_mask = (preds == y)
        total_correct += correct_mask.sum().item()
        total_samples += y.size(0)

        # Update Group Stats
        for k in range(NUM_GROUPS):
            mask = (g == k)
            if mask.any():
                group_correct[k] += correct_mask[mask].sum().item()
                group_total[k] += mask.sum().item()

        # Update Confusion Matrix
        for t, p in zip(y.view(-1), preds.view(-1)):
            cm[t.long(), p.long()] += 1

    # Calculate Accuracies
    group_accs = (group_correct / group_total.clamp(min=1)).cpu().numpy()
    overall_acc = total_correct / total_samples
    worst_group_acc = group_accs.min()

    # Print Group Results
    print(f"\n>>> Detailed Results for {model_name} <<<")
    print(f"{'Group Name':<20} | {'Size':<6} | {'Accuracy':<10}")
    print("-" * 42)
    group_names = ["Land (Land bg)", "Land (Water bg)", "Water (Land bg)", "Water (Water bg)"]
    for i, name in enumerate(group_names):
        print(f"{name:<20} | {int(group_total[i]):<6} | {group_accs[i]*100:.2f}%")
    print("-" * 42)
    print(f"{'Overall Accuracy':<20} | {overall_acc*100:.2f}%")
    print(f"{'Worst-Group Acc':<20} | {worst_group_acc*100:.2f}%")

    # Print Confusion Matrix
    print(f"\n[Confusion Matrix] (Rows=True, Cols=Pred)")
    print(f"{'':<12} | {'Pred Land':<10} | {'Pred Water':<10}")
    print("-" * 38)
    print(f"{'True Land':<12} | {cm[0,0]:<10} | {cm[0,1]:<10}")
    print(f"{'True Water':<12} | {cm[1,0]:<10} | {cm[1,1]:<10}")

    return overall_acc, worst_group_acc

# Run Evaluation
print("\n========================================")
print(" FINAL DETAILED PAPER TABLE (With CM)")
print("========================================")

# 1. Evaluate IRS
'''
get_detailed_metrics(best_model_irs, loaders['test'], "IRS-Group")
'''

In [ ]:
# =========================
# SETUP: Data Loading
# =========================
# This cell MUST be run before any training!
# It sets up: loaders, P_global_group, group_counts

set_seed(SEED)

print("Using DATA_ROOT:", DATA_ROOT)
print("Preparing data...")
loaders, P_global_group = get_loaders_and_priors()

# Extract group counts for GroupDRO
train_ds = loaders["train"].dataset
group_counts = train_ds.meta["group"].value_counts().sort_index().values.tolist()
print(f"Group counts: {group_counts}")
print(f"P_global_group: {P_global_group}")
print(f"\n✅ Data ready! loaders, P_global_group, and group_counts are now available.")

In [ ]:
# ==========================================
# === ALGORITHM CONTROL FLAGS ===
# ==========================================
# Control which algorithms to run for each phase.
# Set to True to run, False to skip.
# This gives you full control - checkpoints are NOT used for gating.
# ==========================================

# --- HYPERPARAMETER SWEEP FLAGS ---
# Set True to run LR/hyperparam sweep for that algorithm
RUN_SWEEP = {
    "ERM-SGD":       False,  # ✅ COMPLETED: best_lr=0.01, valWorst=0.6391
    "ERM":           False,  # ✅ COMPLETED: best_lr=1e-05, valWorst=0.5789 (Adam)
    "SAM":           False,  # ✅ COMPLETED: best_lr=1e-05, valWorst=0.5489
    "IRS-Group":     False,  # ✅ COMPLETED: best_lr=0.0001, valWorst=0.8421
    "KL-RS":         False,  # ✅ COMPLETED: best_lr=0.0001, alt_iters=2, valWorst=0.6502
    "V-REx":         False,  # ✅ COMPLETED: best_lr=0.0001, rex_beta=10.0, valWorst=0.6316
    "MM-REx":        False,  # ✅ COMPLETED: best_lr=0.0001, rex_lambda=1.0, valWorst=0.6015
    "IRMv1":         False,  # ✅ COMPLETED: best_lr=0.0001, penalty_weight=100, valWorst=0.6015
    "GroupDRO":      False,  # ✅ COMPLETED: best_lr=0.0001, alpha=0.2, valWorst=0.7218
    "χ²-DRO":        False,  # ✅ COMPLETED: best_lr=0.0001, rho=1.0, valWorst=0.6692
    "CVaR-DRO":      False,   # Run sweep - NOT YET COMPLETED
}

# --- FULL TRAINING FLAGS ---
# Set True to run full training (using best hyperparams from sweep or defaults)
RUN_FULL_TRAINING = {
    "ERM-SGD":       True,
    "ERM":           True,
    "SAM":           True,
    "IRS-Group":     True,    # Keep IRS enabled by default
    "KL-RS":         True,
    "V-REx":         True,
    "MM-REx":        True,
    "IRMv1":         True,
    "GroupDRO":      True,
    "χ²-DRO":        True,
    "CVaR-DRO":      True,
}

# --- TEST EVALUATION FLAGS ---
# Set True to evaluate on test set (only meaningful if model was trained)
RUN_TEST_EVAL = {
    "ERM-SGD":       True,
    "ERM":           True,
    "SAM":           True,
    "IRS-Group":     True,    # Keep IRS enabled by default
    "KL-RS":         True,
    "V-REx":         True,
    "MM-REx":        True,
    "IRMv1":         True,
    "GroupDRO":      True,
    "χ²-DRO":        True,
    "CVaR-DRO":      True,
}

# --- ANALYSIS FLAGS ---
# Set True to include in final analysis/visualization
RUN_ANALYSIS = {
    "ERM-SGD":       True,
    "ERM":           True,
    "SAM":           True,
    "IRS-Group":     True,    # Keep IRS enabled by default
    "KL-RS":         True,
    "V-REx":         True,
    "MM-REx":        True,
    "IRMv1":         True,
    "GroupDRO":      True,
    "χ²-DRO":        True,
    "CVaR-DRO":      True,
}

# === HELPER FUNCTIONS ===
def should_sweep(algo: str) -> bool:
    """Check if we should run hyperparameter sweep for this algorithm."""
    return RUN_SWEEP.get(algo, False)

def should_train(algo: str) -> bool:
    """Check if we should run full training for this algorithm."""
    return RUN_FULL_TRAINING.get(algo, False)

def should_test(algo: str) -> bool:
    """Check if we should evaluate on test set for this algorithm."""
    return RUN_TEST_EVAL.get(algo, False)

def should_analyze(algo: str) -> bool:
    """Check if we should include in analysis for this algorithm."""
    return RUN_ANALYSIS.get(algo, False)

def print_run_plan():
    """Print which algorithms will be run in each phase."""
    algos = list(RUN_SWEEP.keys())

    print("="*70)
    print(" ALGORITHM RUN PLAN")
    print("="*70)
    print(f"{'Algorithm':<15} | {'Sweep':<8} | {'Train':<8} | {'Test':<8} | {'Analysis':<8}")
    print("-"*70)
    for algo in algos:
        sweep = "✅" if should_sweep(algo) else "❌"
        train = "✅" if should_train(algo) else "❌"
        test = "✅" if should_test(algo) else "❌"
        analyze = "✅" if should_analyze(algo) else "❌"
        print(f"{algo:<15} | {sweep:<8} | {train:<8} | {test:<8} | {analyze:<8}")
    print("="*70)

# Print the plan
print_run_plan()

In [ ]:
# ==========================================
# === Utility Functions ===
# ==========================================

import gc

def cuda_clear_cache():
    """Clear CUDA cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def reset_seeds(seed: int = SEED):
    """Reset random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

def log_epoch(tag, ep, tr_loss, val_worst, val_overall, t0, extra=""):
    """Log epoch progress."""
    elapsed = time.time() - t0
    hrs, rem = divmod(elapsed, 3600)
    mins, secs = divmod(rem, 60)
    t_str = f"{int(hrs):02d}:{int(mins):02d}:{int(secs):02d}"
    msg = f"[{tag}] ep{ep:03d} trainLoss={tr_loss:.4f} valWorst={val_worst:.4f} valOverall={val_overall:.4f} t={t_str}"
    if extra:
        msg += f" | {extra}"
    print(msg)

In [ ]:
# ==========================================
# === SAM Optimizer (Foret et al., ICLR 2021) ===
# ==========================================

class SAMSGD(torch.optim.SGD):
    """
    Sharpness-Aware Minimization with SGD optimizer.
    Paper: "Sharpness-Aware Minimization for Efficiently Improving Generalization" (ICLR 2021)
    """
    def __init__(self, params, lr=1e-3, rho=0.05, **kwargs):
        if rho <= 0:
            raise ValueError(f"rho must be positive, got {rho}")
        self.rho = float(rho)
        super().__init__(params, lr=lr, **kwargs)

    @torch.no_grad()
    def _grad_norm(self):
        device_ = self.param_groups[0]["params"][0].device
        norms = []
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is not None:
                    norms.append(p.grad.detach().norm(2))
        if not norms:
            return torch.tensor(0., device=device_)
        return torch.norm(torch.stack(norms), p=2)

    @torch.no_grad()
    def _epsilon(self, scale):
        epsilons = []
        for group in self.param_groups:
            eps_group = []
            for p in group["params"]:
                if p.grad is None:
                    eps_group.append(None)
                    continue
                e = p.grad * scale
                p.add_(e)
                eps_group.append(e)
            epsilons.append(eps_group)
        return epsilons

    @torch.no_grad()
    def _restore(self, epsilons):
        for group, eps_group in zip(self.param_groups, epsilons):
            for p, e in zip(group["params"], eps_group):
                if e is not None:
                    p.sub_(e)

def disable_running_stats(model):
    """Freeze BN running stats for SAM's second forward pass."""
    for m in model.modules():
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
            m._sam_backup_momentum = m.momentum
            m.momentum = 0.0

def enable_running_stats(model):
    """Restore BN momentum after SAM."""
    for m in model.modules():
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm) and hasattr(m, "_sam_backup_momentum"):
            m.momentum = m._sam_backup_momentum
            delattr(m, "_sam_backup_momentum")

In [ ]:
# ==========================================
# === DRO Loss Functions ===
# ==========================================

def irm_penalty(logits, y):
    """
    IRMv1 penalty: ||∇_w [ loss(w ∘ Φ(x); y) ] ||^2 at w=1.0
    Paper: Arjovsky et al., "Invariant Risk Minimization" (2019)
    """
    scale = torch.ones(1, requires_grad=True, device=logits.device, dtype=logits.dtype)
    loss = nn.CrossEntropyLoss()(logits * scale, y)
    grad = torch.autograd.grad(loss, [scale], create_graph=True)[0]
    return (grad ** 2).sum()


def get_rex_penalty(losses, mode="vrex", lam=1.0):
    """
    Compute variance-based or Min-Max objective from environment losses.
    Reference: Krueger et al. "Out-of-Distribution Generalization via Risk Extrapolation (REx)" ICML 2021
    """
    if mode == "vrex":
        mean_loss = sum(losses) / len(losses)
        var_loss = sum((l - mean_loss) ** 2 for l in losses) / len(losses)
        return var_loss
    elif mode == "mmrex":
        loss_max = max(losses)
        loss_min = min(losses)
        return lam * loss_max + (1 - lam) * loss_min
    else:
        raise ValueError(f"Unknown REx mode: {mode}")


def chi2_dro_loss(loss_vec: torch.Tensor, rho: float,
                  bisect_tol: float = 1e-5, max_iter: int = 50) -> torch.Tensor:
    """
    Chi-squared DRO loss.
    R_ρ(θ) = inf_η { √(1+2ρ) · √(E_P[(ℓ(θ;X) - η)_+²]) + η }
    """
    B = loss_vec.size(0)
    if B == 0:
        return loss_vec.new_tensor(0.0)

    rho = max(float(rho), 0.0)
    c = math.sqrt(1.0 + 2.0 * rho)

    eta_min = loss_vec.min().item()
    eta_max = loss_vec.max().item()

    if abs(eta_max - eta_min) < 1e-12:
        return loss_vec.mean()

    def eval_R(eta):
        diff = loss_vec - eta
        positive_part = torch.clamp(diff, min=0.0)
        var_term = (positive_part ** 2).mean()
        return c * torch.sqrt(var_term + 1e-12) + eta

    for _ in range(max_iter):
        if (eta_max - eta_min) < bisect_tol:
            break
        eta_mid = 0.5 * (eta_min + eta_max)
        diff = loss_vec - eta_mid
        positive_part = torch.clamp(diff, min=0.0)
        var_term = (positive_part ** 2).mean()
        grad = 1.0 - c * (positive_part.mean()) / torch.sqrt(var_term + 1e-12)
        if grad.item() < 0:
            eta_min = eta_mid
        else:
            eta_max = eta_mid

    eta_star = 0.5 * (eta_min + eta_max)
    return eval_R(eta_star)


def cvar_loss_from_batch(loss_vec: torch.Tensor, alpha: float) -> torch.Tensor:
    """
    CVaR-DRO loss from Levy et al., "Large-Scale Methods for Distributionally Robust Optimization" (NeurIPS 2020).
    CVaR_α(ℓ) = mean of top-α quantile losses.
    """
    alpha = max(min(float(alpha), 1.0), 1e-6)
    B = loss_vec.size(0)
    k = max(1, int(math.ceil(alpha * B)))
    topk_losses, _ = torch.topk(loss_vec, k, largest=True, sorted=False)
    return topk_losses.mean()

In [ ]:
# ==========================================
# === GroupDRO LossComputer (Sagawa et al., 2020) ===
# ==========================================

class LossComputer:
    """
    Paper-faithful GroupDRO loss computer from Sagawa et al., "Distributionally Robust Neural Networks
    for Group Shifts: On the Importance of Regularization for Worst-Case Generalization" (ICLR 2020).
    """
    def __init__(self, group_counts, alpha=0.2, gamma=0.1, device='cuda'):
        self.alpha = alpha
        self.gamma = gamma
        self.device = device
        self.n_groups = len(group_counts)

        self.adv_probs = torch.ones(self.n_groups, device=device) / self.n_groups
        self.exp_avg_loss = torch.zeros(self.n_groups, device=device)
        self.exp_avg_initialized = torch.zeros(self.n_groups, device=device).bool()
        self.group_counts = torch.tensor(group_counts, dtype=torch.float32, device=device)
        self.group_frac = self.group_counts / self.group_counts.sum()

    def loss(self, yhat, y, group_idx, is_training=True):
        per_sample_losses = nn.CrossEntropyLoss(reduction='none')(yhat, y)

        group_losses = []
        group_counts_batch = []
        for g in range(self.n_groups):
            mask = (group_idx == g)
            if mask.any():
                group_loss = per_sample_losses[mask].mean()
                group_losses.append(group_loss)
                group_counts_batch.append(mask.sum().item())
            else:
                group_losses.append(torch.tensor(0.0, device=self.device))
                group_counts_batch.append(0)

        group_losses = torch.stack(group_losses)
        group_counts_batch = torch.tensor(group_counts_batch, dtype=torch.float32, device=self.device)

        if is_training:
            for g in range(self.n_groups):
                if group_counts_batch[g] > 0:
                    if not self.exp_avg_initialized[g]:
                        self.exp_avg_loss[g] = group_losses[g].detach()
                        self.exp_avg_initialized[g] = True
                    else:
                        self.exp_avg_loss[g] = (
                            self.gamma * group_losses[g].detach() +
                            (1 - self.gamma) * self.exp_avg_loss[g]
                        )

            adv_probs = torch.exp(self.alpha * self.exp_avg_loss.detach())
            adv_probs = adv_probs / (adv_probs.sum() + 1e-12)
            self.adv_probs = adv_probs

        weighted_loss = (self.adv_probs * group_losses).sum()
        return weighted_loss, {"avg_loss": per_sample_losses.mean().item()}

In [ ]:
# ==========================================
# === Training Functions (ERM, SAM) ===
# ==========================================

def train_erm(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
              weight_decay=0.0, print_every=5):
    """Standard ERM training with Adam optimizer."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_fn = nn.CrossEntropyLoss()

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"ERM Ep {ep}"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            logits = model(x)
            loss = ce_fn(logits, y)
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())
            is_best = " *BEST*"
        else:
            is_best = ""

        if ep % print_every == 0:
            log_epoch("ERM", ep, tr_loss, val_worst, val_overall, t0)
            print(f"  {is_best}")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist


def train_erm_sgd(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
                  weight_decay=0.0, momentum=0.9, print_every=5):
    """Standard ERM training with SGD optimizer."""
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=momentum)
    ce_fn = nn.CrossEntropyLoss()

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"ERM-SGD Ep {ep}"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            logits = model(x)
            loss = ce_fn(logits, y)
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())
            is_best = " *BEST*"
        else:
            is_best = ""

        if ep % print_every == 0:
            log_epoch("ERM-SGD", ep, tr_loss, val_worst, val_overall, t0)
            print(f"  {is_best}")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist


def train_sam(model, train_loader, val_loader, *, epochs=50, lr=1e-4, rho=0.05,
              weight_decay=0.0, print_every=5):
    """SAM training with SGD base optimizer."""
    model.to(device)
    opt = SAMSGD(model.parameters(), lr=lr, rho=rho, weight_decay=weight_decay)
    ce_fn = nn.CrossEntropyLoss()

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"SAM Ep {ep}"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            # First forward-backward
            enable_running_stats(model)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = ce_fn(logits, y)
            loss.backward()

            # Compute perturbation
            gnorm = opt._grad_norm()
            scale = rho / (gnorm + 1e-12)
            epsilons = opt._epsilon(scale)

            # Second forward-backward
            disable_running_stats(model)
            opt.zero_grad(set_to_none=True)
            logits_pert = model(x)
            loss_pert = ce_fn(logits_pert, y)
            loss_pert.backward()

            # Restore and step
            opt._restore(epsilons)
            enable_running_stats(model)
            opt.step()

            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())
            is_best = " *BEST*"
        else:
            is_best = ""

        if ep % print_every == 0:
            log_epoch("SAM", ep, tr_loss, val_worst, val_overall, t0)
            print(f"  {is_best}")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist


@torch.no_grad()
def eval_overall_accuracy(model, loader):
    """Compute overall accuracy."""
    model.eval()
    correct, total = 0, 0
    for x, y, _, g in loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / max(1, total)

In [ ]:
# ==========================================
# === Training Functions (V-REx, MM-REx, IRMv1) ===
# ==========================================
# Paper-faithful implementations with proper environment handling and penalty annealing

def train_vrex(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
               rex_beta=1.0, penalty_anneal_epochs=10, weight_decay=0.0, print_every=5):
    """
    V-REx training (Krueger et al., ICML 2021).

    L = ERM + β * Var(R_e)

    NOTE: Uses environment (background) labels, NOT all 4 groups.
    Environment is extracted as: env = g % 2 (0=land, 1=water background)

    Includes penalty annealing as recommended in paper.
    """
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_none = nn.CrossEntropyLoss(reduction='none')

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        # Anneal penalty weight (paper recommendation)
        penalty_weight = min(1.0, float(ep) / float(penalty_anneal_epochs)) if ep <= penalty_anneal_epochs else 1.0

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"V-REx Ep {ep}"):
            x, y, g = x.to(device), y.to(device), g.to(device)

            # Extract environment from group (place = g % 2)
            env = g % 2  # 0 = land background, 1 = water background

            opt.zero_grad(set_to_none=True)

            logits = model(x)
            ce_vec = ce_none(logits, y)

            # Compute per-ENVIRONMENT losses (2 environments, not 4 groups)
            env_losses = []
            for e in [0, 1]:
                mask = (env == e)
                if mask.any():
                    env_losses.append(ce_vec[mask].mean())

            if len(env_losses) == 2:
                # V-REx: L = ERM + β * Var(R_e)
                mean_loss = sum(env_losses) / len(env_losses)
                var_penalty = get_rex_penalty(env_losses, mode="vrex")
                loss = mean_loss + penalty_weight * rex_beta * var_penalty
            else:
                # If only one environment in batch, use mean loss
                loss = ce_vec.mean()

            loss.backward()
            opt.step()

            total_loss += ce_vec.mean().item() * x.size(0)  # Track ERM loss
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())

        if ep % print_every == 0:
            log_epoch("V-REx", ep, tr_loss, val_worst, val_overall, t0)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist


def train_mmrex(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
                rex_lambda=1.5, penalty_anneal_epochs=10, weight_decay=0.0, print_every=5):
    """
    MM-REx training (Krueger et al., ICML 2021).

    L = λ * max(R_e) + (1-λ) * min(R_e)

    This REPLACES the ERM objective (not a penalty).
    - λ = 0.5: ERM (equal weight to min/max)
    - λ = 1.0: Worst-case only (like GroupDRO)
    - λ > 1.0: Extrapolation beyond worst-case (key insight of MM-REx)

    NOTE: Uses environment labels, anneals λ from 0.5 to target.
    """
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_none = nn.CrossEntropyLoss(reduction='none')

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        # Anneal λ from 0.5 (ERM) to target λ (paper recommendation)
        anneal_factor = min(1.0, float(ep) / float(penalty_anneal_epochs)) if ep <= penalty_anneal_epochs else 1.0
        current_lambda = 0.5 + anneal_factor * (rex_lambda - 0.5)

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"MM-REx Ep {ep}"):
            x, y, g = x.to(device), y.to(device), g.to(device)

            # Extract environment from group
            env = g % 2

            opt.zero_grad(set_to_none=True)

            logits = model(x)
            ce_vec = ce_none(logits, y)

            # Compute per-ENVIRONMENT losses
            env_losses = []
            for e in [0, 1]:
                mask = (env == e)
                if mask.any():
                    env_losses.append(ce_vec[mask].mean())

            if len(env_losses) == 2:
                # MM-REx: L = λ*max(R_e) + (1-λ)*min(R_e)
                loss = get_rex_penalty(env_losses, mode="mmrex", lam=current_lambda)
            else:
                loss = ce_vec.mean()

            loss.backward()
            opt.step()

            total_loss += ce_vec.mean().item() * x.size(0)
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())

        if ep % print_every == 0:
            log_epoch("MM-REx", ep, tr_loss, val_worst, val_overall, t0)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist


def train_irm(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
              penalty_weight=10000.0, penalty_anneal_epochs=20, weight_decay=0.0, print_every=5):
    """
    IRMv1 training (Arjovsky et al., 2019).

    L = ERM + λ * Σ_e ||∇_w [R_e(w·Φ)]|w=1||²

    NOTE: Uses environment (background) labels for invariance.
    Includes penalty annealing as strongly recommended in original paper.

    Default penalty_weight=10000 and penalty_anneal_epochs=20 from paper.
    """
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        # Anneal penalty (critical for IRM convergence)
        current_penalty_weight = penalty_weight * min(1.0, float(ep) / float(penalty_anneal_epochs))

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"IRMv1 Ep {ep}"):
            x, y, g = x.to(device), y.to(device), g.to(device)

            # Extract environment from group
            env = g % 2

            opt.zero_grad(set_to_none=True)

            logits = model(x)

            # Compute per-ENVIRONMENT losses and IRM penalties
            env_losses = []
            env_penalties = []
            for e in [0, 1]:
                mask = (env == e)
                if mask.sum() > 1:  # Need at least 2 samples for gradient
                    env_logits = logits[mask]
                    env_y = y[mask]
                    env_loss = nn.CrossEntropyLoss()(env_logits, env_y)
                    env_pen = irm_penalty(env_logits, env_y)
                    env_losses.append(env_loss)
                    env_penalties.append(env_pen)

            if len(env_losses) == 2:
                mean_loss = sum(env_losses) / len(env_losses)
                mean_penalty = sum(env_penalties) / len(env_penalties)
                loss = mean_loss + current_penalty_weight * mean_penalty
            else:
                loss = nn.CrossEntropyLoss()(logits, y)

            loss.backward()
            opt.step()

            total_loss += nn.CrossEntropyLoss()(logits.detach(), y).item() * x.size(0)
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())

        if ep % print_every == 0:
            log_epoch("IRMv1", ep, tr_loss, val_worst, val_overall, t0)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist

print("✅ V-REx, MM-REx, IRMv1 defined (paper-faithful with env labels & penalty annealing)")

In [ ]:
# ==========================================
# === Training Functions (GroupDRO, χ²-DRO, CVaR-DRO) ===
# ==========================================

def train_groupdro(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
                   alpha=0.2, weight_decay=0.0, print_every=5, group_counts=None):
    """GroupDRO training (Sagawa et al., ICLR 2020)."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    if group_counts is None:
        group_counts = [1] * NUM_GROUPS
    loss_computer = LossComputer(group_counts, alpha=alpha, device=device)

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"GroupDRO Ep {ep}"):
            x, y, g = x.to(device), y.to(device), g.to(device)
            opt.zero_grad(set_to_none=True)

            logits = model(x)
            loss, _ = loss_computer.loss(logits, y, g, is_training=True)
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())

        if ep % print_every == 0:
            log_epoch("GroupDRO", ep, tr_loss, val_worst, val_overall, t0)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist


def train_chi2_dro(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
                   rho=0.1, weight_decay=0.0, print_every=5):
    """χ²-DRO training."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_none = nn.CrossEntropyLoss(reduction='none')

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"χ²-DRO Ep {ep}"):
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)

            logits = model(x)
            ce_vec = ce_none(logits, y)
            loss = chi2_dro_loss(ce_vec, rho)
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())

        if ep % print_every == 0:
            log_epoch("χ²-DRO", ep, tr_loss, val_worst, val_overall, t0)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist


def train_cvar_dro(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
                   alpha=0.1, weight_decay=0.0, print_every=5):
    """CVaR-DRO training (Levy et al., NeurIPS 2020)."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_none = nn.CrossEntropyLoss(reduction='none')

    best_val_worst = -1.0
    best_model_state = None

    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, y, _, g in tqdm(train_loader, leave=False, desc=f"CVaR-DRO Ep {ep}"):
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)

            logits = model(x)
            ce_vec = ce_none(logits, y)
            loss = cvar_loss_from_batch(ce_vec, alpha)
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)

        tr_loss = total_loss / max(1, total_n)
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)

        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)

        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())

        if ep % print_every == 0:
            log_epoch("CVaR-DRO", ep, tr_loss, val_worst, val_overall, t0)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return tr_hist, va_worst_hist, va_overall_hist

In [ ]:
# ==========================================
# === KL-RS Training Function (revised) ===
# ==========================================
# Faithful to Yan et al. 2024 (arXiv:2408.09157) §4.1, Algorithms 1 & 2.
#
#   Algorithm 1 (feasibility(λ)): apply stochastic optimization to solve
#       min_θ  E_P̂[ exp((l(θ; z) - τ) / λ) ]
#     and return whether f* := achieved objective is ≤ 1.
#
#   Algorithm 2 (bisect λ): doubling phase until feasibility, then bisect
#     over [λ_lo (infeasible), λ_hi (feasible)]. Each probe is a call to
#     Algorithm 1.
#
# Outer alt_iters interleaves an explicit θ-step (more training at the
# current λ) with one round of Algorithm 2.  All training is warm-started
# across probes through a single persistent Adam optimizer — this is the
# standard vision-experiment relaxation of the paper's "min_θ" oracle.
#
# Fixes vs the previous cell 14:
#   1. Single persistent Adam optimizer (the old version reinstantiated
#      Adam on every _feasibility_oracle call, wiping the (m, v) buffers).
#   2. inner_epochs_probe defaulted to 3 (was 1) so each probe trains long
#      enough for the achieved f* to be a meaningful feasibility signal.
#   3. Gradient clipping (max_norm=1.0) on the exp-tilted loss by default
#      to tame the +exp_clamp saturation when λ is small relative to losses.
#   4. Best-validation checkpoint (by worst-group accuracy) is tracked
#      throughout and restored before return.

import math
import copy
import time
import torch
import torch.nn as nn

def train_klrs(model, train_loader, val_loader, *, epochs=50, lr=1e-4,
               weight_decay=0.0, base_tau=None, warmup_epochs=1,
               alt_iters=2, inner_epochs_theta=None, inner_epochs_probe=3,
               lam_init=1.0, lam_min=1e-6, lam_max=1024.0,
               max_doubling=6, max_bisect=4,
               bisect_tol=1e-2, expand_factor=2.0,
               exp_clamp=50.0, grad_clip_max_norm=1.0,
               print_every=5, **_compat_unused):
    """KL-RS (Yan et al., 2024) — paper-faithful, alt_iters wrapper kept.

    See cell header for algorithm + bug-fix summary.
    """
    model.to(device)
    ce_nored = nn.CrossEntropyLoss(reduction="none")
    ce_mean = nn.CrossEntropyLoss(reduction="mean")

    if inner_epochs_theta is None:
        inner_epochs_theta = max(1, (epochs - warmup_epochs) // max(1, alt_iters))

    # τ: fixed (provided or estimated as 1.05× mean CE on a few batches).
    if base_tau is None:
        with torch.no_grad():
            model.eval()
            est, n = 0.0, 0
            for xb, yb, *_ in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                est += ce_mean(model(xb), yb).item()
                n += 1
                if n >= 5: break
            tau = 1.05 * (est / max(n, 1))
    else:
        tau = float(base_tau)

    print(f"[KLRS] τ = {tau:.4f}  |  alt_iters={alt_iters}  "
          f"inner_theta={inner_epochs_theta}  inner_probe={inner_epochs_probe}")

    # Single persistent Adam: never reinstantiated across probes.
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_worst, best_model_state = -1.0, None
    tr_hist, va_worst_hist, va_overall_hist = [], [], []
    t0 = time.time()
    global_ep = 0

    def _remaining_epochs():
        return max(0, int(epochs) - global_ep)

    def _log(tag, lam_val, tr_loss):
        nonlocal best_val_worst, best_model_state
        _, val_worst = eval_metrics(model, val_loader)
        val_overall = eval_overall_accuracy(model, val_loader)
        tr_hist.append(tr_loss)
        va_worst_hist.append(val_worst)
        va_overall_hist.append(val_overall)
        is_best = ""
        if val_worst > best_val_worst:
            best_val_worst = val_worst
            best_model_state = copy.deepcopy(model.state_dict())
            is_best = " *BEST*"
        if global_ep == 1 or global_ep % print_every == 0:
            elapsed = time.time() - t0
            lam_str = f"  λ={lam_val:.4f}" if lam_val is not None else ""
            print(f"[{tag} ep {global_ep:3d}] trLoss={tr_loss:.4f}  "
                  f"valWorst={val_worst:.4f}  valOvr={val_overall:.4f}"
                  f"{lam_str}  [{elapsed:.1f}s]{is_best}")

    def _train_one_epoch(loss_kind, lam_val=None):
        model.train()
        total_loss, total_n = 0.0, 0
        for xb, yb, *_ in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            if loss_kind == "erm":
                loss = ce_mean(model(xb), yb)
            else:  # klrs: minimize E[ exp((ce - τ) / λ) ]  (paper Algorithm 1)
                ce_vec = ce_nored(model(xb), yb)
                a = (ce_vec - tau) / lam_val
                a = torch.clamp(a, min=-exp_clamp, max=exp_clamp)
                loss = torch.exp(a).mean()
            loss.backward()
            if grad_clip_max_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_max_norm)
            opt.step()
            total_loss += loss.item() * xb.size(0)
            total_n += xb.size(0)
        return total_loss / max(1, total_n)

    @torch.no_grad()
    def _estimate_f_star(lam_val):
        """f* := E_P̂[ exp((l - τ) / λ) ] on the current (trained) θ."""
        model.eval()
        tot, n_batches = 0.0, 0
        for xb, yb, *_ in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            ce_vec = ce_nored(model(xb), yb)
            a = (ce_vec - tau) / lam_val
            a = torch.clamp(a, min=-exp_clamp, max=exp_clamp)
            tot += torch.exp(a).mean().item()
            n_batches += 1
        return tot / max(1, n_batches)

    def _probe(lam_val, tag):
        """Algorithm 1: warm-start train inner_epochs_probe epochs at λ; return f*."""
        nonlocal global_ep
        probe_epochs = min(inner_epochs_probe, _remaining_epochs())
        for _ in range(probe_epochs):
            global_ep += 1
            tr_loss = _train_one_epoch("klrs", lam_val)
            _log(tag, lam_val, tr_loss)
        return _estimate_f_star(lam_val)

    def _algorithm2(start_lam, tag_prefix):
        """Algorithm 2: doubling + bisection over λ; warm-started probes."""
        # Doubling phase: increase λ until feasibility.
        lam_hi = max(start_lam, lam_min)
        f_hi = _probe(lam_hi, f"{tag_prefix}-A1")
        if _remaining_epochs() <= 0:
            return lam_hi
        lam_lo = 0.0
        for _ in range(max_doubling):
            if f_hi <= 1.0:
                break
            lam_lo = lam_hi
            lam_hi = min(lam_hi * expand_factor, lam_max)
            f_hi = _probe(lam_hi, f"{tag_prefix}-A1")
            if _remaining_epochs() <= 0:
                return lam_hi
        if f_hi > 1.0:
            print(f"  [{tag_prefix}] doubling exhausted at λ={lam_hi:g} (f*={f_hi:.4f}).")
            return lam_hi

        # If feasible at start_lam already, establish an infeasible lower bound
        # by halving once. If still feasible, accept the lower λ.
        if lam_lo == 0.0:
            cand = max(lam_hi / expand_factor, lam_min)
            f_cand = _probe(cand, f"{tag_prefix}-A1")
            if _remaining_epochs() <= 0:
                return cand
            if f_cand <= 1.0:
                print(f"  [{tag_prefix}] λ ≤ {cand:g} already feasible; using λ={cand:g}.")
                return cand
            lam_lo = cand

        # Bisection in [lam_lo (infeasible), lam_hi (feasible)].
        for b in range(max_bisect):
            if (lam_hi - lam_lo) / max(1.0, lam_hi) < bisect_tol:
                break
            lam_mid = (lam_lo + lam_hi) / 2.0
            f_mid = _probe(lam_mid, f"{tag_prefix}-A1")
            if _remaining_epochs() <= 0:
                return lam_mid
            if f_mid <= 1.0:
                lam_hi = lam_mid
            else:
                lam_lo = lam_mid
        return lam_hi

    # ===============================
    # Phase 1: ERM warmup
    # ===============================
    for _ in range(min(warmup_epochs, _remaining_epochs())):
        global_ep += 1
        tr_loss = _train_one_epoch("erm")
        _log("KLRS-warmup", None, tr_loss)

    # ===============================
    # Phase 2: alt_iters alternations of θ-step + Algorithm 2
    # ===============================
    lam = float(lam_init)
    for it in range(1, alt_iters + 1):
        if _remaining_epochs() <= 0:
            break
        # (a) θ-step at current λ.
        for _ in range(min(inner_epochs_theta, _remaining_epochs())):
            global_ep += 1
            tr_loss = _train_one_epoch("klrs", lam)
            _log(f"KLRS-it{it}-theta", lam, tr_loss)

        # (b) λ-step via Algorithm 2 (warm-started training inside).
        if _remaining_epochs() <= 0:
            break
        new_lam = _algorithm2(lam, f"KLRS-it{it}")
        print(f"  [KLRS-it{it}] λ-search: {lam:.4f} -> {new_lam:.4f}")
        lam = new_lam

    # Restore best-validation checkpoint.
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"[KLRS] Restored best-val checkpoint (val_worst={best_val_worst:.4f}).")

    return tr_hist, va_worst_hist, va_overall_hist


print("OK  train_klrs defined (paper-faithful: Alg 1 + Alg 2, single Adam, longer probes, best-val ckpt)")


In [ ]:
# ==========================================
# === Hyperparameter Grids ===
# ==========================================

import itertools

# Base LR grid for all algorithms (4 values)
LR_GRID = [1e-5, 1e-4, 1e-3, 1e-2]

# SAM rho
SAM_RHO = 0.05

# KL-RS alt_iters grid (3 values)
KLRS_ALT_ITERS_GRID = [2, 5, 10]

# IRM penalty weight grid (paper uses 10000 as default, sweep around it)
IRM_PENALTY_GRID = [100, 1000, 10000, 100000]
IRM_PENALTY_ANNEAL_EPOCHS = 20  # Paper default

# REx grids (paper-faithful)
VREX_BETA_GRID = [0.1, 0.5, 1.0, 5.0, 10.0]  # 5 values
MMREX_LAMBDA_GRID = [1.0, 1.5, 2.0, 3.0]      # 4 values (λ > 1 extrapolates beyond worst-case)
REX_PENALTY_ANNEAL_EPOCHS = 10  # Paper recommendation

# GroupDRO grids (3 values)
GROUPDRO_ALPHA_GRID = [0.1, 0.2, 0.5]

# χ²-DRO grids
CHI2_RHO_GRID = [0.01, 0.1, 0.5, 1.0, 2.0]  # 5 values

# CVaR-DRO grids - Levy et al. (NeurIPS 2020)
CVAR_ALPHA_GRID = [0.05, 0.1, 0.2, 0.3, 0.5]  # 5 values

# === EPOCH SETTINGS ===
VAL_SWEEP_EPOCHS = 30   # Epochs for hyperparameter sweeps
FULL_TRAIN_EPOCHS = 50  # Epochs for full training

# Shared tau for IRS and KL-RS
RS_SHARED_TAU = 0.1

print("="*60)
print("Hyperparameter grids defined (paper-faithful).")
print("="*60)
print(f"LR Grid: {LR_GRID}")
print(f"KL-RS alt_iters Grid: {KLRS_ALT_ITERS_GRID}")
print(f"IRM Penalty Grid: {IRM_PENALTY_GRID} (anneal_epochs={IRM_PENALTY_ANNEAL_EPOCHS})")
print(f"V-REx Beta Grid: {VREX_BETA_GRID} (anneal_epochs={REX_PENALTY_ANNEAL_EPOCHS})")
print(f"MM-REx Lambda Grid: {MMREX_LAMBDA_GRID} (anneal_epochs={REX_PENALTY_ANNEAL_EPOCHS})")
print(f"GroupDRO Alpha Grid: {GROUPDRO_ALPHA_GRID}")
print(f"χ²-DRO Rho Grid: {CHI2_RHO_GRID}")
print(f"CVaR-DRO Alpha Grid: {CVAR_ALPHA_GRID}")
print("="*60)
print(f"Sweep Epochs: {VAL_SWEEP_EPOCHS}")
print(f"Full Training Epochs: {FULL_TRAIN_EPOCHS}")
print("="*60)
print("\nGrid sizes (LR × Hyperparam):")
print(f"  ERM:      {len(LR_GRID)} runs (LR only)")
print(f"  SAM:      {len(LR_GRID)} runs (LR only)")
print(f"  IRS:      {len(LR_GRID)} runs (LR only)")
print(f"  KL-RS:    {len(LR_GRID)} × {len(KLRS_ALT_ITERS_GRID)} = {len(LR_GRID) * len(KLRS_ALT_ITERS_GRID)} runs")
print(f"  V-REx:    {len(LR_GRID)} × {len(VREX_BETA_GRID)} = {len(LR_GRID) * len(VREX_BETA_GRID)} runs")
print(f"  MM-REx:   {len(LR_GRID)} × {len(MMREX_LAMBDA_GRID)} = {len(LR_GRID) * len(MMREX_LAMBDA_GRID)} runs")
print(f"  IRMv1:    {len(LR_GRID)} × {len(IRM_PENALTY_GRID)} = {len(LR_GRID) * len(IRM_PENALTY_GRID)} runs")
print(f"  GroupDRO: {len(LR_GRID)} × {len(GROUPDRO_ALPHA_GRID)} = {len(LR_GRID) * len(GROUPDRO_ALPHA_GRID)} runs")
print(f"  χ²-DRO:   {len(LR_GRID)} × {len(CHI2_RHO_GRID)} = {len(LR_GRID) * len(CHI2_RHO_GRID)} runs")
print(f"  CVaR-DRO: {len(LR_GRID)} × {len(CVAR_ALPHA_GRID)} = {len(LR_GRID) * len(CVAR_ALPHA_GRID)} runs")

In [ ]:
# ==========================================
# === Hyperparameter Sweep Utilities ===
# ==========================================

def lr_sweep(algo_name: str, lr_list, train_fn, train_kwargs: dict,
             train_epochs: int = None, seed: int = SEED, print_every_sweep: int = 10):
    """
    Sweep over learning rates and select the best one by worst-group validation accuracy.
    Uses VAL_SWEEP_EPOCHS by default.
    """
    if train_epochs is None:
        train_epochs = VAL_SWEEP_EPOCHS  # Use global default (30)

    results = []
    print(f"[{algo_name}] Starting LR sweep: {len(lr_list)} configurations, {train_epochs} epochs each")

    for lr in lr_list:
        reset_seeds(seed)
        cuda_clear_cache()
        model = fresh_model()
        local_kwargs = dict(train_kwargs)
        local_kwargs["print_every"] = print_every_sweep

        try:
            out = train_fn(model, loaders["train"], loaders["val"], epochs=train_epochs, lr=lr, **local_kwargs)
        except Exception as e:
            print(f"[SWEEP:{algo_name}] lr={lr:g}  ❌ {e}")
            continue

        # Extract worst-group accuracy (second element in returns)
        va_worst = out[1] if len(out) >= 2 else []
        val_worst_final = float(va_worst[-1]) if va_worst else float("nan")

        if not np.isfinite(val_worst_final):
            print(f"[SWEEP:{algo_name}] lr={lr:g}  ❌ valWorst is NaN/Inf")
            continue

        results.append({"lr": lr, "val_worst_acc": val_worst_final})
        print(f"[SWEEP:{algo_name}] lr={lr:g}  valWorst(final)={val_worst_final:.4f}")

        del model
        cuda_clear_cache()

    best = max(results, key=lambda r: r["val_worst_acc"]) if results else {"lr": None, "val_worst_acc": float("nan")}
    print(f"[SWEEP:{algo_name}] ★ best_lr={best['lr']}  best_valWorst={best['val_worst_acc']:.4f}")
    return best["lr"], results


def hyperparam_sweep(algo_name: str, config_grid, train_fn, base_train_kwargs: dict,
                     train_epochs: int = None, seed: int = SEED, print_every_sweep: int = 10):
    """
    Sweep over a grid of hyperparameter configurations (LR × other params).
    Uses VAL_SWEEP_EPOCHS by default.
    """
    if train_epochs is None:
        train_epochs = VAL_SWEEP_EPOCHS  # Use global default (30)

    results = []
    print(f"[{algo_name}] Starting hyperparam sweep: {len(config_grid)} configurations, {train_epochs} epochs each")

    for i, config in enumerate(config_grid):
        reset_seeds(seed)
        cuda_clear_cache()
        model = fresh_model()

        local_kwargs = dict(base_train_kwargs)
        local_kwargs.update(config)
        local_kwargs["print_every"] = print_every_sweep

        epochs = local_kwargs.pop("epochs", train_epochs)
        config_str = ", ".join([f"{k}={v}" for k, v in config.items()])

        try:
            out = train_fn(model, loaders["train"], loaders["val"], epochs=epochs, **local_kwargs)
        except Exception as e:
            print(f"[SWEEP:{algo_name}] [{i+1}/{len(config_grid)}] {config_str}  ❌ {e}")
            del model
            cuda_clear_cache()
            continue

        va_worst = out[1] if len(out) >= 2 else []
        val_worst_final = float(va_worst[-1]) if va_worst else float("nan")

        if not np.isfinite(val_worst_final):
            print(f"[SWEEP:{algo_name}] [{i+1}/{len(config_grid)}] {config_str}  ❌ valWorst is NaN/Inf")
            del model
            cuda_clear_cache()
            continue

        result_entry = dict(config)
        result_entry["val_worst_acc"] = val_worst_final
        results.append(result_entry)
        print(f"[SWEEP:{algo_name}] [{i+1}/{len(config_grid)}] {config_str}  valWorst={val_worst_final:.4f}")

        del model
        cuda_clear_cache()

    if not results:
        print(f"[SWEEP:{algo_name}] No valid results!")
        return None, []

    best = max(results, key=lambda r: r["val_worst_acc"])
    best_config = {k: v for k, v in best.items() if k != "val_worst_acc"}
    best_str = ", ".join([f"{k}={v}" for k, v in best_config.items()])
    print(f"[SWEEP:{algo_name}] ★ BEST: {best_str}  valWorst={best['val_worst_acc']:.4f}")
    return best_config, results

print(f"✅ Sweep utilities defined (default sweep epochs: {VAL_SWEEP_EPOCHS})")

In [ ]:
# ==========================================
# === RUN HYPERPARAMETER SWEEPS ===
# ==========================================
# Only runs algorithms where RUN_SWEEP[algo] = True
# Grid search uses Cartesian product: LR × other_params (e.g., 4×3 = 12 runs)

# Storage for best hyperparameters (with paper-faithful defaults)
# Updated from completed sweeps
best_hyperparams = {
    "ERM-SGD": {"lr": 0.01},   # ✅ From sweep: valWorst=0.6391
    "ERM": {"lr": 1e-5},       # ✅ From sweep: valWorst=0.5789 (Adam)
    "SAM": {"lr": 1e-5, "rho": SAM_RHO},  # ✅ From sweep: valWorst=0.5489
    "IRS-Group": {"lr": 1e-4},  # ✅ From sweep: valWorst=0.8421
    "KL-RS": {"lr": 1e-4, "alt_iters": 2, "base_tau": RS_SHARED_TAU},  # ✅ From sweep: valWorst=0.6502
    "V-REx": {"lr": 1e-4, "rex_beta": 10.0},     # ✅ From sweep: valWorst=0.6316
    "MM-REx": {"lr": 1e-4, "rex_lambda": 1.0},   # ✅ From sweep: valWorst=0.6015
    "IRMv1": {"lr": 1e-4, "penalty_weight": 100.0},  # ✅ From sweep: valWorst=0.6015
    "GroupDRO": {"lr": 1e-4, "alpha": 0.2},      # ✅ From sweep: valWorst=0.7218
    "χ²-DRO": {"lr": 1e-4, "rho": 1.0},          # ✅ From sweep: valWorst=0.6692
    "CVaR-DRO": {"lr": 1e-4, "alpha": 0.05},      # Default - will be swept
}

print("="*60)
print("SWEEP CONFIGURATION")
print("="*60)
print(f"Sweep epochs: {VAL_SWEEP_EPOCHS}")
print(f"Full training epochs: {FULL_TRAIN_EPOCHS}")
print(f"LR grid: {LR_GRID}")
print("="*60)

# --- ERM-SGD LR Sweep ---
if should_sweep("ERM-SGD"):
    print("\n" + "="*60)
    print(f"▶ LR sweep: ERM-SGD ({len(LR_GRID)} runs)")
    print("="*60)
    best_erm_sgd_lr, _ = lr_sweep("ERM-SGD", LR_GRID, train_erm_sgd, dict(weight_decay=WEIGHT_DECAY))
    best_hyperparams["ERM-SGD"]["lr"] = best_erm_sgd_lr or 1e-4

# --- ERM LR Sweep ---
if should_sweep("ERM"):
    print("\n" + "="*60)
    print(f"▶ LR sweep: ERM ({len(LR_GRID)} runs)")
    print("="*60)
    best_erm_lr, _ = lr_sweep("ERM", LR_GRID, train_erm, dict(weight_decay=WEIGHT_DECAY))
    best_hyperparams["ERM"]["lr"] = best_erm_lr or 1e-4

# --- SAM LR Sweep ---
if should_sweep("SAM"):
    print("\n" + "="*60)
    print(f"▶ LR sweep: SAM ({len(LR_GRID)} runs)")
    print("="*60)
    best_sam_lr, _ = lr_sweep("SAM", LR_GRID, train_sam, dict(rho=SAM_RHO, weight_decay=WEIGHT_DECAY))
    best_hyperparams["SAM"]["lr"] = best_sam_lr or 1e-4

# --- IRS-Group LR Sweep ---
if should_sweep("IRS-Group"):
    print("\n" + "="*60)
    print(f"▶ LR sweep: IRS-Group ({len(LR_GRID)} runs)")
    print("="*60)

    # Custom wrapper to adapt train_algorithm to lr_sweep interface
    def train_irs_for_sweep(model, train_loader, val_loader, *, epochs, lr, weight_decay, print_every=10):
        """
        Wrapper to use train_algorithm for sweeps.
        train_algorithm: (algo_name, model, loaders, P_global_group, epochs, lr, warmup_epochs=0, alpha=0.2) -> model
        lr_sweep expects: (model, train_loader, val_loader, epochs, lr, ...) -> (tr_hist, val_worst_hist, val_overall_hist)
        """
        # Reconstruct loaders dict for train_algorithm
        loaders_dict = {"train": train_loader, "val": val_loader}

        # Call train_algorithm with correct signature
        trained_model = train_algorithm(
            algo_name="IRS-Group",
            model=model,
            loaders=loaders_dict,
            P_global_group=P_global_group,  # Use global variable
            epochs=epochs,
            lr=lr,
            warmup_epochs=0,  # No warmup for sweep
            alpha=0.2  # Not used for IRS-Group
        )

        # train_algorithm returns model and tracks best internally
        # We need to evaluate to get final worst-group accuracy for lr_sweep
        val_loss, val_worst = eval_metrics(trained_model, val_loader)

        # Return format expected by lr_sweep: (tr_hist, val_worst_hist, val_overall_hist)
        # Since train_algorithm doesn't track history, return lists with final values
        return [], [val_worst], []

    best_irs_lr, _ = lr_sweep("IRS-Group", LR_GRID, train_irs_for_sweep, dict(weight_decay=WEIGHT_DECAY))
    best_hyperparams["IRS-Group"]["lr"] = best_irs_lr or 1e-5

# --- KL-RS Sweep (LR × alt_iters) ---
if should_sweep("KL-RS"):
    print("\n" + "="*60)
    print(f"▶ Hyperparam sweep: KL-RS (LR × alt_iters = {len(LR_GRID)} × {len(KLRS_ALT_ITERS_GRID)} = {len(LR_GRID) * len(KLRS_ALT_ITERS_GRID)} runs)")
    print("="*60)

    # For KL-RS, we adjust inner_epochs_theta based on total epochs
    def train_klrs_for_sweep(model, train_loader, val_loader, *, epochs, lr, alt_iters, weight_decay, print_every=10):
        # Calculate inner_epochs_theta so that total training time ≈ epochs
        # Total work ≈ warmup + alt_iters * inner_epochs_theta + remaining
        warmup = 3
        inner_epochs_theta = max(1, (epochs - warmup) // alt_iters)
        return train_klrs(
            model, train_loader, val_loader,
            epochs=epochs, lr=lr, alt_iters=alt_iters,
            inner_epochs_theta=inner_epochs_theta,
            warmup_epochs=warmup,
            base_tau=RS_SHARED_TAU,
            weight_decay=weight_decay,
            print_every=print_every
        )

    klrs_config_grid = [
        {"lr": lr, "alt_iters": ai}
        for lr, ai in itertools.product(LR_GRID, KLRS_ALT_ITERS_GRID)
    ]
    best_klrs_config, _ = hyperparam_sweep("KL-RS", klrs_config_grid, train_klrs_for_sweep, dict(weight_decay=WEIGHT_DECAY))
    if best_klrs_config:
        best_hyperparams["KL-RS"].update(best_klrs_config)

# --- V-REx Sweep (LR × beta) ---
if should_sweep("V-REx"):
    print("\n" + "="*60)
    print(f"▶ Hyperparam sweep: V-REx (LR × Beta = {len(LR_GRID)} × {len(VREX_BETA_GRID)} = {len(LR_GRID) * len(VREX_BETA_GRID)} runs)")
    print("="*60)
    vrex_config_grid = [
        {"lr": lr, "rex_beta": beta}
        for lr, beta in itertools.product(LR_GRID, VREX_BETA_GRID)
    ]
    best_vrex_config, _ = hyperparam_sweep("V-REx", vrex_config_grid, train_vrex, dict(weight_decay=WEIGHT_DECAY))
    if best_vrex_config:
        best_hyperparams["V-REx"].update(best_vrex_config)

# --- MM-REx Sweep (LR × lambda) ---
if should_sweep("MM-REx"):
    print("\n" + "="*60)
    print(f"▶ Hyperparam sweep: MM-REx (LR × Lambda = {len(LR_GRID)} × {len(MMREX_LAMBDA_GRID)} = {len(LR_GRID) * len(MMREX_LAMBDA_GRID)} runs)")
    print("="*60)
    mmrex_config_grid = [
        {"lr": lr, "rex_lambda": lam}
        for lr, lam in itertools.product(LR_GRID, MMREX_LAMBDA_GRID)
    ]
    best_mmrex_config, _ = hyperparam_sweep("MM-REx", mmrex_config_grid, train_mmrex, dict(weight_decay=WEIGHT_DECAY))
    if best_mmrex_config:
        best_hyperparams["MM-REx"].update(best_mmrex_config)

# --- IRMv1 Sweep (LR × penalty) ---
if should_sweep("IRMv1"):
    print("\n" + "="*60)
    print(f"▶ Hyperparam sweep: IRMv1 (LR × Penalty = {len(LR_GRID)} × {len(IRM_PENALTY_GRID)} = {len(LR_GRID) * len(IRM_PENALTY_GRID)} runs)")
    print("="*60)
    irm_config_grid = [
        {"lr": lr, "penalty_weight": pw}
        for lr, pw in itertools.product(LR_GRID, IRM_PENALTY_GRID)
    ]
    best_irm_config, _ = hyperparam_sweep("IRMv1", irm_config_grid, train_irm, dict(weight_decay=WEIGHT_DECAY))
    if best_irm_config:
        best_hyperparams["IRMv1"].update(best_irm_config)

# --- GroupDRO Sweep (LR × alpha) ---
if should_sweep("GroupDRO"):
    print("\n" + "="*60)
    print(f"▶ Hyperparam sweep: GroupDRO (LR × Alpha = {len(LR_GRID)} × {len(GROUPDRO_ALPHA_GRID)} = {len(LR_GRID) * len(GROUPDRO_ALPHA_GRID)} runs)")
    print("="*60)
    groupdro_config_grid = [
        {"lr": lr, "alpha": alpha}
        for lr, alpha in itertools.product(LR_GRID, GROUPDRO_ALPHA_GRID)
    ]
    best_groupdro_config, _ = hyperparam_sweep("GroupDRO", groupdro_config_grid, train_groupdro,
                                                dict(weight_decay=WEIGHT_DECAY, group_counts=group_counts))
    if best_groupdro_config:
        best_hyperparams["GroupDRO"].update(best_groupdro_config)

# --- χ²-DRO Sweep (LR × rho) ---
if should_sweep("χ²-DRO"):
    print("\n" + "="*60)
    print(f"▶ Hyperparam sweep: χ²-DRO (LR × Rho = {len(LR_GRID)} × {len(CHI2_RHO_GRID)} = {len(LR_GRID) * len(CHI2_RHO_GRID)} runs)")
    print("="*60)
    chi2_config_grid = [
        {"lr": lr, "rho": rho}
        for lr, rho in itertools.product(LR_GRID, CHI2_RHO_GRID)
    ]
    best_chi2_config, _ = hyperparam_sweep("χ²-DRO", chi2_config_grid, train_chi2_dro, dict(weight_decay=WEIGHT_DECAY))
    if best_chi2_config:
        best_hyperparams["χ²-DRO"].update(best_chi2_config)

# --- CVaR-DRO Sweep (LR × alpha) ---
if should_sweep("CVaR-DRO"):
    print("\n" + "="*60)
    print(f"▶ Hyperparam sweep: CVaR-DRO (LR × Alpha = {len(LR_GRID)} × {len(CVAR_ALPHA_GRID)} = {len(LR_GRID) * len(CVAR_ALPHA_GRID)} runs)")
    print("="*60)
    cvar_config_grid = [
        {"lr": lr, "alpha": alpha}
        for lr, alpha in itertools.product(LR_GRID, CVAR_ALPHA_GRID)
    ]
    best_cvar_config, _ = hyperparam_sweep("CVaR-DRO", cvar_config_grid, train_cvar_dro, dict(weight_decay=WEIGHT_DECAY))
    if best_cvar_config:
        best_hyperparams["CVaR-DRO"].update(best_cvar_config)

print("\n" + "="*60)
print("✅ Hyperparameter sweeps completed!")
print("="*60)
print("\nBest hyperparameters:")
for algo, params in best_hyperparams.items():
    print(f"  {algo}: {params}")

In [ ]:
# ==========================================
# === RUN FULL TRAINING ===
# ==========================================
# Only runs algorithms where RUN_FULL_TRAINING[algo] = True
# IRS uses the ORIGINAL working code (train_algorithm function)
# Full training uses FULL_TRAIN_EPOCHS (50 epochs)

# Storage for trained models and results
trained_models = {}
results = {}
wallclock_times = {}  # Track wallclock time for each algorithm

print("="*60)
print("FULL TRAINING CONFIGURATION")
print("="*60)
print(f"Full training epochs: {FULL_TRAIN_EPOCHS}")
print("="*60)

# --- ERM-SGD Full Training ---
if should_train("ERM-SGD"):
    print("\n" + "="*60)
    print(f"▶ Full training: ERM-SGD ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_erm_sgd = fresh_model()
    t_start = time.time()
    tr_erm_sgd, va_worst_erm_sgd, va_overall_erm_sgd = train_erm_sgd(
        model_erm_sgd, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["ERM-SGD"]["lr"], weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["ERM-SGD"] = time.time() - t_start
    trained_models["ERM-SGD"] = model_erm_sgd
    results["ERM-SGD"] = {"tr": tr_erm_sgd, "va_worst": va_worst_erm_sgd, "va_overall": va_overall_erm_sgd}

# --- ERM Full Training ---
if should_train("ERM"):
    print("\n" + "="*60)
    print(f"▶ Full training: ERM ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_erm = fresh_model()
    t_start = time.time()
    tr_erm, va_worst_erm, va_overall_erm = train_erm(
        model_erm, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["ERM"]["lr"], weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["ERM"] = time.time() - t_start
    trained_models["ERM"] = model_erm
    results["ERM"] = {"tr": tr_erm, "va_worst": va_worst_erm, "va_overall": va_overall_erm}

# --- SAM Full Training ---
if should_train("SAM"):
    print("\n" + "="*60)
    print(f"▶ Full training: SAM ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_sam = fresh_model()
    t_start = time.time()
    tr_sam, va_worst_sam, va_overall_sam = train_sam(
        model_sam, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["SAM"]["lr"], rho=SAM_RHO, weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["SAM"] = time.time() - t_start
    trained_models["SAM"] = model_sam
    results["SAM"] = {"tr": tr_sam, "va_worst": va_worst_sam, "va_overall": va_overall_sam}

# --- IRS-Group Full Training (USES ORIGINAL WORKING CODE!) ---
if should_train("IRS-Group"):
    print("\n" + "="*60)
    print(f"▶ Full training: IRS-Group ({FULL_TRAIN_EPOCHS} epochs) [ORIGINAL WORKING CODE]")
    print("="*60)
    reset_seeds(SEED)
    model_irs = fresh_model()
    t_start = time.time()
    # Uses the ORIGINAL train_algorithm function from the working IRS code!
    best_model_irs = train_algorithm(
        "IRS-Group", model_irs, loaders, P_global_group,
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["IRS-Group"]["lr"], warmup_epochs=WARMUP_EPOCHS
    )
    wallclock_times["IRS-Group"] = time.time() - t_start
    trained_models["IRS-Group"] = best_model_irs
    results["IRS-Group"] = {"tr": [], "va_worst": [], "va_overall": []}  # Logged during training

# --- KL-RS Full Training ---
if should_train("KL-RS"):
    print("\n" + "="*60)
    print(f"▶ Full training: KL-RS ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_klrs = fresh_model()
    t_start = time.time()
    best_alt_iters = best_hyperparams["KL-RS"].get("alt_iters", 10)
    warmup = 3
    inner_epochs_theta = max(1, (FULL_TRAIN_EPOCHS - warmup) // best_alt_iters)
    tr_klrs, va_worst_klrs, va_overall_klrs = train_klrs(
        model_klrs, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["KL-RS"]["lr"],
        alt_iters=best_alt_iters,
        inner_epochs_theta=inner_epochs_theta,
        warmup_epochs=warmup,
        base_tau=RS_SHARED_TAU,
        weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["KL-RS"] = time.time() - t_start
    trained_models["KL-RS"] = model_klrs
    results["KL-RS"] = {"tr": tr_klrs, "va_worst": va_worst_klrs, "va_overall": va_overall_klrs}

# --- V-REx Full Training ---
if should_train("V-REx"):
    print("\n" + "="*60)
    print(f"▶ Full training: V-REx ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_vrex = fresh_model()
    t_start = time.time()
    tr_vrex, va_worst_vrex, va_overall_vrex = train_vrex(
        model_vrex, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["V-REx"]["lr"],
        rex_beta=best_hyperparams["V-REx"]["rex_beta"], weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["V-REx"] = time.time() - t_start
    trained_models["V-REx"] = model_vrex
    results["V-REx"] = {"tr": tr_vrex, "va_worst": va_worst_vrex, "va_overall": va_overall_vrex}

# --- MM-REx Full Training ---
if should_train("MM-REx"):
    print("\n" + "="*60)
    print(f"▶ Full training: MM-REx ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_mmrex = fresh_model()
    t_start = time.time()
    tr_mmrex, va_worst_mmrex, va_overall_mmrex = train_mmrex(
        model_mmrex, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["MM-REx"]["lr"],
        rex_lambda=best_hyperparams["MM-REx"]["rex_lambda"], weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["MM-REx"] = time.time() - t_start
    trained_models["MM-REx"] = model_mmrex
    results["MM-REx"] = {"tr": tr_mmrex, "va_worst": va_worst_mmrex, "va_overall": va_overall_mmrex}

# --- IRMv1 Full Training ---
if should_train("IRMv1"):
    print("\n" + "="*60)
    print(f"▶ Full training: IRMv1 ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_irm = fresh_model()
    t_start = time.time()
    tr_irm, va_worst_irm, va_overall_irm = train_irm(
        model_irm, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["IRMv1"]["lr"],
        penalty_weight=best_hyperparams["IRMv1"]["penalty_weight"], weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["IRMv1"] = time.time() - t_start
    trained_models["IRMv1"] = model_irm
    results["IRMv1"] = {"tr": tr_irm, "va_worst": va_worst_irm, "va_overall": va_overall_irm}

# --- GroupDRO Full Training ---
if should_train("GroupDRO"):
    print("\n" + "="*60)
    print(f"▶ Full training: GroupDRO ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_groupdro = fresh_model()
    t_start = time.time()
    tr_groupdro, va_worst_groupdro, va_overall_groupdro = train_groupdro(
        model_groupdro, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["GroupDRO"]["lr"],
        alpha=best_hyperparams["GroupDRO"]["alpha"], weight_decay=WEIGHT_DECAY,
        group_counts=group_counts, print_every=5
    )
    wallclock_times["GroupDRO"] = time.time() - t_start
    trained_models["GroupDRO"] = model_groupdro
    results["GroupDRO"] = {"tr": tr_groupdro, "va_worst": va_worst_groupdro, "va_overall": va_overall_groupdro}

# --- χ²-DRO Full Training ---
if should_train("χ²-DRO"):
    print("\n" + "="*60)
    print(f"▶ Full training: χ²-DRO ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_chi2 = fresh_model()
    t_start = time.time()
    tr_chi2, va_worst_chi2, va_overall_chi2 = train_chi2_dro(
        model_chi2, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["χ²-DRO"]["lr"],
        rho=best_hyperparams["χ²-DRO"]["rho"], weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["χ²-DRO"] = time.time() - t_start
    trained_models["χ²-DRO"] = model_chi2
    results["χ²-DRO"] = {"tr": tr_chi2, "va_worst": va_worst_chi2, "va_overall": va_overall_chi2}

# --- CVaR-DRO Full Training ---
if should_train("CVaR-DRO"):
    print("\n" + "="*60)
    print(f"▶ Full training: CVaR-DRO ({FULL_TRAIN_EPOCHS} epochs)")
    print("="*60)
    reset_seeds(SEED)
    model_cvar = fresh_model()
    t_start = time.time()
    tr_cvar, va_worst_cvar, va_overall_cvar = train_cvar_dro(
        model_cvar, loaders["train"], loaders["val"],
        epochs=FULL_TRAIN_EPOCHS, lr=best_hyperparams["CVaR-DRO"]["lr"],
        alpha=best_hyperparams["CVaR-DRO"]["alpha"], weight_decay=WEIGHT_DECAY, print_every=5
    )
    wallclock_times["CVaR-DRO"] = time.time() - t_start
    trained_models["CVaR-DRO"] = model_cvar
    results["CVaR-DRO"] = {"tr": tr_cvar, "va_worst": va_worst_cvar, "va_overall": va_overall_cvar}

print("\n" + "="*60)
print("✅ Full training completed!")
print(f"Trained models: {list(trained_models.keys())}")
print("="*60)

# Print wallclock times
print("\n" + "="*60)
print("WALLCLOCK TIMES (Full Training)")
print("="*60)
for algo, elapsed in wallclock_times.items():
    mins, secs = divmod(elapsed, 60)
    hrs, mins = divmod(mins, 60)
    print(f"  {algo:<15}: {int(hrs):02d}:{int(mins):02d}:{secs:05.2f}")
print("="*60)

In [ ]:
# ==========================================
# === TEST SET EVALUATION ===
# ==========================================
# Only evaluates algorithms where RUN_TEST_EVAL[algo] = True

# Define helper function first
@torch.no_grad()
def get_per_group_accuracy(model, loader):
    """Get per-group accuracy."""
    model.eval()
    group_correct = torch.zeros(NUM_GROUPS)
    group_total = torch.zeros(NUM_GROUPS)

    for x, y, _, g in loader:
        x, y, g = x.to(device), y.to(device), g.to(device)
        preds = model(x).argmax(1)
        correct = (preds == y)
        for k in range(NUM_GROUPS):
            mask = (g == k)
            if mask.any():
                group_correct[k] += correct[mask].sum().item()
                group_total[k] += mask.sum().item()

    return (group_correct / group_total.clamp(min=1)).numpy()

# Run test evaluation
test_results = {}
GROUP_NAMES = ["Land (Land bg)", "Land (Water bg)", "Water (Land bg)", "Water (Water bg)"]

print("\n" + "="*60)
print(" FINAL TEST EVALUATION")
print(" (Best validation models evaluated on test set)")
print("="*60)

for algo_name in trained_models.keys():
    if not should_test(algo_name):
        continue

    model = trained_models[algo_name]
    test_loss, test_worst = eval_metrics(model, loaders["test"])
    test_overall = eval_overall_accuracy(model, loaders["test"])

    # Get per-group accuracies
    group_accs = get_per_group_accuracy(model, loaders["test"])

    test_results[algo_name] = {
        "worst_group_acc": test_worst,
        "overall_acc": test_overall,
        "per_group_acc": group_accs,
    }

    print(f"\n>>> {algo_name} <<<")
    print(f"Overall Accuracy: {test_overall*100:.2f}%")
    print(f"Worst-Group Accuracy: {test_worst*100:.2f}%")
    print("Per-Group Accuracy:")
    for i, gname in enumerate(GROUP_NAMES):
        print(f"  {gname}: {group_accs[i]*100:.2f}%")

# Summary table
if test_results:
    print("\n" + "="*60)
    print(" SUMMARY TABLE (Test Set)")
    print("="*60)
    print(f"{'Method':<18} | {'Worst-Group':<12} | {'Overall':<10}")
    print("-" * 48)
    for algo_name, r in test_results.items():
        print(f"{algo_name:<18} | {r['worst_group_acc']*100:>10.2f}% | {r['overall_acc']*100:>8.2f}%")
    print("="*60)

In [ ]:
# ==========================================
# === VISUALIZATION & ANALYSIS ===
# ==========================================
# Only includes algorithms where RUN_ANALYSIS[algo] = True

import matplotlib.pyplot as plt

def _ep(n):
    return list(range(1, n + 1))

# Get algorithms to analyze
algos_to_analyze = [algo for algo in results.keys() if should_analyze(algo)]

if algos_to_analyze:
    print("\n" + "="*60)
    print(" VISUALIZATION")
    print(f" Algorithms: {algos_to_analyze}")
    print("="*60)

    # Plot 1: Training Loss
    fig, ax = plt.subplots(figsize=(10, 6))
    for algo in algos_to_analyze:
        if "tr" in results[algo] and results[algo]["tr"]:
            y = results[algo]["tr"]
            ax.plot(_ep(len(y)), y, label=algo)
    ax.set_title("Training Loss vs Epoch — Waterbirds", fontsize=14)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.legend(loc='best')
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    # Plot 2: Validation Worst-Group Accuracy (KEY METRIC)
    fig, ax = plt.subplots(figsize=(10, 6))
    for algo in algos_to_analyze:
        if "va_worst" in results[algo] and results[algo]["va_worst"]:
            y = results[algo]["va_worst"]
            ax.plot(_ep(len(y)), y, label=algo)
    ax.set_title("Validation Worst-Group Accuracy vs Epoch — Waterbirds", fontsize=14)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Worst-Group Accuracy")
    ax.legend(loc='best')
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    # Plot 3: Validation Overall Accuracy
    fig, ax = plt.subplots(figsize=(10, 6))
    for algo in algos_to_analyze:
        if "va_overall" in results[algo] and results[algo]["va_overall"]:
            y = results[algo]["va_overall"]
            ax.plot(_ep(len(y)), y, label=algo)
    ax.set_title("Validation Overall Accuracy vs Epoch — Waterbirds", fontsize=14)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Overall Accuracy")
    ax.legend(loc='best')
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

else:
    print("\n[INFO] No algorithms selected for analysis. Set RUN_ANALYSIS[algo] = True to include.")